In [1]:
# imports
from importlib.metadata import version # for version
import re # for regex
import tiktoken # for BPE

# Print library version
print("tiktoken version:", version("tiktoken"))


tiktoken version: 0.7.0


In [2]:
# load data
with open("../data/the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
print("Total number of character:", len(raw_text))
print(raw_text[:100])

Total number of character: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no g


In [3]:
# simple tokenizer
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(len(preprocessed))

4690


In [4]:
preprocessed[4685:]

['kind', 'of', 'art', '.', '"']

In [5]:
print(preprocessed[:30])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


In [6]:
# Converting tokens into token IDs
all_tokens = sorted(set(preprocessed))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])
vocab = {token:integer for integer,token in enumerate(all_tokens)}
print(len(vocab.items()))
# inv_vocab = {integer:token for integer,token in enumerate(all_words)}
# for i, item in enumerate(vocab.items()):
#     print(item)
#     if i > 10:
#         break

for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

1132
('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


In [7]:
# Tokenizer - 1st Attempt
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids]) 
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [9]:
# tester
tokenizer = SimpleTokenizerV1(vocab)
text = """"It's the last he painted, you know," Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print(ids)
print(tokenizer.decode(ids))

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]
" It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


In [10]:
# Tokenizer - 2nd Attempt
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items()}
        
    def encode(self, text):
        preprocessed = re.split(r'([,.?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [item if item in self.str_to_int
                        else "<|unk|>" for item in preprocessed]
 
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
 
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [32]:
text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."
text = " <|endoftext|> ".join((text1, text2))
print(text)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [34]:
# tester
tokenizer = SimpleTokenizerV2(vocab)
# text = """"It's the last he painted, you know," Mrs. Gisburn said with pardonable pride."""
text = "Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace."
ids = tokenizer.encode(text)
print(ids)
text = (tokenizer.decode(ids))
print(text)

[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 1131, 7]
<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of <|unk|>.
The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


In [12]:
# Tokenizer - 3rd Attempt
tokenizer = tiktoken.get_encoding("gpt2")

In [13]:
# tester
text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."
# text = " <|endoftext|> ".join((text1, text2))
# text = "Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace."
text = "Akwirw ier"
print(text)
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)
for item in integers:
    print(f'tokenId: {item}, token:{tokenizer.decode([item])}')
strings = tokenizer.decode(integers)
print(strings)

Akwirw ier
[33901, 86, 343, 86, 220, 959]
tokenId: 33901, token:Ak
tokenId: 86, token:w
tokenId: 343, token:ir
tokenId: 86, token:w
tokenId: 220, token: 
tokenId: 959, token:ier
Akwirw ier


### Tokenize the whole dataset

In [14]:
with open("../data/the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
 
enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

5145


In [15]:
# we remove the first 50 tokens from the dataset for demonstration purposes 
# as it results in a slightly more interesting text passage in the next steps
enc_sample = enc_text[50:]

In [16]:
context_size = 4
x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]
print(f"x: {x}")
print(f"y:      {y} ")

x: [290, 4920, 2241, 287]
y:      [4920, 2241, 287, 257] 


In [17]:
# next word prediction

for i in range(1, context_size+1):
    context = enc_sample[:i] # stores until i and i starts from 1; so context will initially have x[0]
    desired = enc_sample[i]
    # print(context, "---->", desired)
    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

 and ---->  established
 and established ---->  himself
 and established himself ---->  in
 and established himself in ---->  a


In [18]:
import torch
torch.__version__
torch.cuda.is_available()
# check if your Mac supports PyTorch acceleration with its Apple Silicon chip
print(torch.backends.mps.is_available())

True


In [19]:
# Create dataset and dataloader that extract chunks from the input text dataset
import torch
from torch.utils.data import Dataset, DataLoader
 
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []
        # Creating tokens form text and generating token ids
        token_ids = tokenizer.encode(txt)
 
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))
 
    def __len__(self):
        return len(self.input_ids)
 
    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

#### A data loader to generate batches with input-with pairs

In [20]:
def create_dataloader_v1(txt, batch_size=4, max_length=256,
        stride=128, shuffle=True, drop_last=True, num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )
 
    return dataloader

In [21]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False)
data_iter = iter(dataloader)
# first_batch = next(data_iter)
# print(first_batch)
# second_batch = next(data_iter)
# print(second_batch)
# third_batch = next(data_iter)
# print(third_batch)

itr = 0
while data_iter and itr < 5:
    batch = next(data_iter)
    print(f"batch_{itr}: {batch}")
    itr += 1

batch_0: [tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]
batch_1: [tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]
batch_2: [tensor([[2885, 1464, 1807, 3619]]), tensor([[1464, 1807, 3619,  402]])]
batch_3: [tensor([[1464, 1807, 3619,  402]]), tensor([[1807, 3619,  402,  271]])]
batch_4: [tensor([[1807, 3619,  402,  271]]), tensor([[ 3619,   402,   271, 10899]])]


In [22]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs: \n", inputs)
print("Targets: \n", targets)


Inputs: 
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
Targets: 
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


### Creating token Embeddings

In [23]:
input_ids = torch.tensor([2, 3, 5, 1])
vocab_size = 6
output_dim = 3

In [24]:
torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


In [25]:
probs = torch.softmax(embedding_layer.weight, dim=1)
probs

tensor([[0.4545, 0.2715, 0.2739],
        [0.2269, 0.4403, 0.3328],
        [0.6819, 0.1558, 0.1622],
        [0.1851, 0.7271, 0.0877],
        [0.1407, 0.6208, 0.2384],
        [0.0770, 0.6011, 0.3219]], grad_fn=<SoftmaxBackward0>)

In [26]:
print(embedding_layer(torch.tensor([3])))

tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)


In [27]:
print(embedding_layer(input_ids))

tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


### Modify Embeddings to encode positional information about a token within a text.

In [28]:
vocab_size = 50257
output_dim = 256
# create weights matrix
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

In [29]:
print(token_embedding_layer(torch.tensor([1])))

tensor([[ 1.2277, -0.4297, -2.2121, -0.3780,  0.9838, -1.0895, -0.6378, -0.6031,
         -1.7753, -0.7490,  0.2781, -0.9621, -0.4223, -1.1036,  0.8398, -1.0029,
         -0.2835, -0.3767, -0.0306, -0.0894, -0.1965, -0.9713,  0.2790, -0.7587,
          1.0669, -0.2985,  0.8558,  1.6098, -1.1893,  1.1677,  0.6220,  2.5737,
         -1.6179,  0.2265, -0.4382,  0.3265, -1.5786, -1.3995,  0.2425,  0.3648,
         -1.1753,  1.7825,  1.7524, -0.2135,  0.4095,  0.0465,  0.5468,  1.1478,
         -0.8614,  0.5338,  0.9376, -0.9225,  0.7047, -0.2722,  0.0781, -0.1134,
          2.3902, -1.4256, -0.4619, -1.5539, -0.3338,  0.2405, -0.0334,  1.5544,
         -0.2936, -1.8027, -0.6933,  1.7409,  0.2698,  0.9595,  0.7744,  1.8721,
          1.0264, -0.5670, -0.2658, -1.1116, -1.3696, -0.6534, -2.1587,  0.8093,
          1.8388, -0.9473,  0.1419,  0.3696, -0.0174, -0.9575,  1.2968,  0.6833,
          0.4343, -0.1340, -2.1467, -1.7984, -0.6822, -0.5191,  0.1669, -1.2620,
         -0.2443,  0.1327,  

In [30]:
max_length = 4
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=max_length, stride=max_length, shuffle=False)
data_iter = iter(dataloader)
batch = inputs, targets = next(data_iter)
print("Token IDs:\n", inputs)
print("\nInputs shape:\n", inputs.shape)

Token IDs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Inputs shape:
 torch.Size([8, 4])


In [31]:
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

torch.Size([8, 4, 256])


In [32]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(context_length))
print(pos_embeddings.shape)

torch.Size([4, 256])


In [33]:
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

torch.Size([8, 4, 256])
